# train_model.ipynb

Train and persist the final IPL win-prediction model.

Reproduces the leakage-safe Phase 3 feature engineering and fits XGBoost on
the chronological training window (seasons through 2023). Saves:

- `ipl_model.pkl` — fitted `XGBClassifier`
- `feature_columns.pkl` — exact training-time feature order

## 1. Imports

In [2]:
import os
import pickle
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier

## 2. Configuration

Paths and constants. The CSV filenames below are the exact names inside `archive (2).zip`.

In [4]:
# Source archive — keep this file next to the notebook
ARCHIVE_ZIP = "archive (2).zip"

# Exact CSV filenames inside the archive (we use the up-to-2025 pair)
DELIVERIES_PATH = "deliveries_updated_ipl_upto_2025.csv"
MATCHES_PATH    = "matches_updated_ipl_upto_2025.csv"

# Chronological cut-off — last training season (inclusive)
TRAIN_END = 2023

# Persistence targets
MODEL_OUT    = "ipl_model.pkl"
FEATURES_OUT = "feature_columns.pkl"

# T20 phase definitions
PP_OVERS, M_OVERS, D_OVERS = 6, 9, 5
PHASE_BINS   = [-1, 5, 14, 19]
PHASE_LABELS = ["Powerplay", "Middle", "Death"]

# Cricket dismissals that count as a wicket against the batting team
WICKET_KINDS = {
    "caught", "bowled", "run out", "lbw", "stumped",
    "caught and bowled", "hit wicket", "obstructing the field",
    "retired hurt", "retired out",
}

# Neutral defaults when a team has no prior history
NEUTRAL_TEAM_STATS = {
    "form_to_date":         0.5,
    "form_last_10":         0.5,
    "avg_runs_scored":      160.0,
    "avg_runs_conceded":    160.0,
    "recent_runs_scored":   160.0,
    "recent_runs_conceded": 160.0,
}
TEAM_COLS = list(NEUTRAL_TEAM_STATS.keys())

## 3. Unpack the archive (only if the CSVs aren't already extracted)

In [5]:
if not (Path(DELIVERIES_PATH).exists() and Path(MATCHES_PATH).exists()):
    with zipfile.ZipFile(ARCHIVE_ZIP) as z:
        z.extractall(".")
    print(f"Extracted CSVs from {ARCHIVE_ZIP}")
else:
    print("CSVs already extracted")

Extracted CSVs from archive (2).zip


## 4. Load + clean

Column-name aliases handle different IPL CSV dump conventions (`inning` vs `innings`, `batsman_runs` vs `runs_off_bat`, etc.). Numeric extras coerce to floats; super-overs and abandoned matches are dropped.

In [6]:
DELIVERY_ALIASES = {
    "match_id":         ["match_id", "matchId", "id"],
    "innings":          ["innings", "inning"],
    "over":             ["over"],
    "ball":             ["ball"],
    "batting_team":     ["batting_team"],
    "bowling_team":     ["bowling_team"],
    "runs_off_bat":     ["runs_off_bat", "batsman_runs", "batter_runs"],
    "extras":           ["extras", "extra_runs"],
    "wides":            ["wides", "isWide", "wide_runs"],
    "noballs":          ["noballs", "isNoBall", "noball_runs"],
    "byes":             ["byes", "Byes", "bye_runs"],
    "legbyes":          ["legbyes", "LegByes", "legbye_runs"],
    "penalty":          ["penalty", "Penalty", "penalty_runs"],
    "wicket_type":      ["wicket_type", "dismissal_kind"],
    "player_dismissed": ["player_dismissed"],
    "date":             ["date", "start_date"],
}
MATCH_ALIASES = {
    "match_id":      ["match_id", "matchId", "id"],
    "date":          ["date", "start_date"],
    "venue":         ["venue"],
    "winner":        ["winner"],
    "team1":         ["team1"],
    "team2":         ["team2"],
    "toss_winner":   ["toss_winner"],
    "toss_decision": ["toss_decision"],
}

def standardize(df, aliases):
    """Rename any alias columns to the canonical name."""
    rename = {}
    for canonical, options in aliases.items():
        for opt in options:
            if opt in df.columns and opt != canonical:
                rename[opt] = canonical
                break
    return df.rename(columns=rename)

deliveries = standardize(pd.read_csv(DELIVERIES_PATH), DELIVERY_ALIASES)
matches    = standardize(pd.read_csv(MATCHES_PATH),    MATCH_ALIASES)

# Coerce numeric extras
for col in ("runs_off_bat", "extras", "wides", "noballs",
            "byes", "legbyes", "penalty"):
    if col in deliveries.columns:
        deliveries[col] = pd.to_numeric(deliveries[col], errors="coerce").fillna(0)
    else:
        deliveries[col] = 0

# Drop super-overs and abandoned matches
deliveries = deliveries[deliveries["innings"].isin([1, 2])].copy()
deliveries["date"] = pd.to_datetime(deliveries["date"], errors="coerce")
matches["date"]    = pd.to_datetime(matches["date"],    errors="coerce")
matches            = matches[matches["winner"].notna()].copy()

print(f"deliveries: {len(deliveries):,} balls   |   matches: {len(matches):,}")

deliveries: 278,034 balls   |   matches: 1,146


## 5. Build the innings-level base table

Pivot ball-by-ball data into one row per `(match_id, innings)` with phase-level runs / wickets and the metadata the feature builders need (teams, venue, toss, date, year, target).


In [7]:
# Phase classification
deliveries["phase"] = pd.Categorical(
    pd.cut(deliveries["over"].astype(int),
           bins=PHASE_BINS, labels=PHASE_LABELS),
    categories=PHASE_LABELS, ordered=True,
)
deliveries["total_runs"] = deliveries["runs_off_bat"] + deliveries["extras"]
deliveries["is_wicket"]  = (deliveries["wicket_type"].astype(str)
                                                      .str.lower()
                                                      .isin(WICKET_KINDS))

# Merge winner + derive target outcome
df = deliveries.merge(matches[["match_id", "winner"]], on="match_id", how="inner")
df["batting_team_won_match"] = df["batting_team"] == df["winner"]
df["year"]              = df["date"].dt.year
df["impact_era_after"]  = (df["year"] >= 2023).astype(int)

# Phase pivots
runs = df.pivot_table(index=["match_id", "innings"], columns="phase",
                      values="total_runs", aggfunc="sum",
                      observed=True).fillna(0)
runs.columns = [f"{str(c).lower()}_runs" for c in runs.columns]

wkts = df.pivot_table(index=["match_id", "innings"], columns="phase",
                      values="is_wicket", aggfunc="sum",
                      observed=True).fillna(0).astype(int)
wkts.columns = [f"{str(c).lower()}_wickets" for c in wkts.columns]

# Innings metadata
meta = (df.groupby(["match_id", "innings"])
          .agg(batting_team=("batting_team", "first"),
               bowling_team=("bowling_team", "first"),
               year=("year", "first"),
               impact_era_after=("impact_era_after", "first"),
               batting_team_won_match=("batting_team_won_match", "first"))
          .reset_index())

mdl = (runs.join(wkts).reset_index()
            .merge(meta, on=["match_id", "innings"])
            .merge(matches[["match_id", "venue", "date",
                            "toss_winner", "toss_decision"]],
                   on="match_id", how="left"))
mdl["batting_first_or_second"] = mdl["innings"]
mdl["target"] = mdl["batting_team_won_match"].astype(int)

# Chronological training mask — referenced by every training-only feature builder below
train_mask = mdl["year"] <= TRAIN_END
print(f"mdl: {len(mdl):,} rows   |   train rows: {train_mask.sum():,}")

mdl: 2,292 rows   |   train rows: 2,010


In [17]:
# =========================
# VALIDATION BLOCK
# =========================

print("Shape of mdl:", mdl.shape)
print("\nColumns:\n", mdl.columns.tolist())

print("\nSample Data:")
print(mdl.head())

print("\nMissing values:\n", mdl.isnull().sum())

print("\nTarget distribution:")
print(mdl["target"].value_counts(normalize=True))

print("\nPhase Runs Summary:")
print(mdl[["powerplay_runs", "middle_runs", "death_runs"]].describe())

print("\nMatch Count Check:")
print("Unique matches in mdl:", mdl["match_id"].nunique())
print("Original matches:", matches["match_id"].nunique())

print("\nTrain vs Test Split:")
print(mdl.groupby(train_mask)["year"].describe())

print("\nCorrelation with target:")
print(mdl.corr(numeric_only=True)["target"].sort_values(ascending=False))

# =========================
# ASSERTIONS (Hard Checks)
# =========================

assert mdl.shape[0] > 0, "❌ mdl is empty"
assert mdl["target"].nunique() == 2, "❌ Target is not binary"
assert mdl[["powerplay_runs", "middle_runs", "death_runs"]].sum().sum() > 0, "❌ Runs are all zero"
assert mdl.isnull().sum().sum() < 1000, "❌ Too many missing values"

print("\n✅ All checks passed successfully!")

Shape of mdl: (2292, 88)

Columns:
 ['match_id', 'innings', 'powerplay_runs', 'middle_runs', 'death_runs', 'powerplay_wickets', 'middle_wickets', 'death_wickets', 'batting_team', 'bowling_team', 'year', 'impact_era_after', 'batting_team_won_match', 'venue', 'date', 'toss_winner', 'toss_decision', 'batting_first_or_second', 'target', 'powerplay_rr', 'middle_rr', 'death_rr', 'cum_runs_after_pp', 'cum_runs_after_15', 'cum_runs_total', 'wkts_after_pp', 'wkts_after_15', 'wkts_total', 'wkts_rem_after_pp', 'wkts_rem_after_15', 'pp_rpw', 'middle_rpw', 'death_rpw', 'pp_collapse', 'middle_collapse', 'death_collapse', 'accel_pp_to_middle', 'accel_middle_to_death', 'death_share_of_innings', 'pp_aggression', 'middle_aggression', 'death_aggression', 'v_pp_avg', 'v_middle_avg', 'v_death_avg', 'v_pp_diff', 'v_middle_diff', 'v_death_diff', 'bat_form_to_date', 'bat_form_last_10', 'bat_avg_runs_scored', 'bat_avg_runs_conceded', 'bat_recent_runs_scored', 'bat_recent_runs_conceded', 'bowl_form_to_date', 'b

## 6. Advanced phase features

Run rates, cumulative resources, runs-per-wicket, collapse flags, acceleration deltas, boundary-pressure proxies. All derive from columns already in `mdl`, so each is observable at the stage where it'd be used.


In [8]:
# Tempo: phase run rates
mdl["powerplay_rr"] = mdl["powerplay_runs"] / PP_OVERS
mdl["middle_rr"]    = mdl["middle_runs"]    / M_OVERS
mdl["death_rr"]     = mdl["death_runs"]     / D_OVERS

# Resource: cumulative runs
mdl["cum_runs_after_pp"] = mdl["powerplay_runs"]
mdl["cum_runs_after_15"] = mdl["powerplay_runs"] + mdl["middle_runs"]
mdl["cum_runs_total"]    = mdl["cum_runs_after_15"] + mdl["death_runs"]

# Resource: cumulative wickets / wickets remaining
mdl["wkts_after_pp"]      = mdl["powerplay_wickets"]
mdl["wkts_after_15"]      = mdl["powerplay_wickets"] + mdl["middle_wickets"]
mdl["wkts_total"]         = mdl["wkts_after_15"] + mdl["death_wickets"]
mdl["wkts_rem_after_pp"]  = (10 - mdl["wkts_after_pp"]).clip(lower=0)
mdl["wkts_rem_after_15"]  = (10 - mdl["wkts_after_15"]).clip(lower=0)

# Stress: runs per wicket lost (denominator capped at 1)
mdl["pp_rpw"]     = mdl["powerplay_runs"] / mdl["powerplay_wickets"].clip(lower=1)
mdl["middle_rpw"] = mdl["middle_runs"]    / mdl["middle_wickets"].clip(lower=1)
mdl["death_rpw"]  = mdl["death_runs"]     / mdl["death_wickets"].clip(lower=1)

# Stress: collapse flags
mdl["pp_collapse"]     = (mdl["powerplay_wickets"] >= 3).astype(int)
mdl["middle_collapse"] = (mdl["middle_wickets"]    >= 3).astype(int)
mdl["death_collapse"]  = (mdl["death_wickets"]     >= 3).astype(int)

# Tempo: acceleration deltas
mdl["accel_pp_to_middle"]    = mdl["middle_rr"] - mdl["powerplay_rr"]
mdl["accel_middle_to_death"] = mdl["death_rr"]  - mdl["middle_rr"]

# Innings shape
mdl["death_share_of_innings"] = mdl["death_runs"] / mdl["cum_runs_total"].clip(lower=1)

# Boundary-pressure proxies (excess scoring above a baseline RR per phase)
mdl["pp_aggression"]     = (mdl["powerplay_runs"] - 6 * PP_OVERS).clip(lower=0)
mdl["middle_aggression"] = (mdl["middle_runs"]    - 7 * M_OVERS).clip(lower=0)
mdl["death_aggression"]  = (mdl["death_runs"]     - 9 * D_OVERS).clip(lower=0)

In [18]:
# =========================
# FEATURE VALIDATION BLOCK
# =========================

print("\nNew Feature Columns Check:")
new_cols = [
    "powerplay_rr", "middle_rr", "death_rr",
    "cum_runs_after_pp", "cum_runs_after_15", "cum_runs_total",
    "wkts_after_pp", "wkts_after_15", "wkts_total",
    "wkts_rem_after_pp", "wkts_rem_after_15",
    "pp_rpw", "middle_rpw", "death_rpw",
    "pp_collapse", "middle_collapse", "death_collapse",
    "accel_pp_to_middle", "accel_middle_to_death",
    "death_share_of_innings",
    "pp_aggression", "middle_aggression", "death_aggression"
]

print([col for col in new_cols if col in mdl.columns])

print("\nSample Data (New Features):")
print(mdl[new_cols].head())

print("\nMissing Values in New Features:")
print(mdl[new_cols].isnull().sum())

print("\nRun Rate Summary:")
print(mdl[["powerplay_rr", "middle_rr", "death_rr"]].describe())

print("\nCumulative Runs Check:")
print(mdl[["cum_runs_after_pp", "cum_runs_after_15", "cum_runs_total"]].describe())

print("\nWickets Check:")
print(mdl[["wkts_total", "wkts_rem_after_pp", "wkts_rem_after_15"]].describe())

print("\nAcceleration Check:")
print(mdl[["accel_pp_to_middle", "accel_middle_to_death"]].describe())

print("\nAggression Metrics Check:")
print(mdl[["pp_aggression", "middle_aggression", "death_aggression"]].describe())

print("\nCollapse Flags Distribution:")
print(mdl[["pp_collapse", "middle_collapse", "death_collapse"]].mean())

# =========================
# ASSERTIONS (CRITICAL)
# =========================

assert mdl[new_cols].isnull().sum().sum() == 0, "❌ Missing values found in new features"
assert mdl["cum_runs_total"].min() >= 0, "❌ Negative total runs detected"
assert mdl["wkts_total"].max() <= 10, "❌ Wickets exceed 10"
assert mdl["death_share_of_innings"].between(0, 1).all(), "❌ Death share outside 0-1 range"

print("\n✅ Feature engineering checks passed successfully!")


New Feature Columns Check:
['powerplay_rr', 'middle_rr', 'death_rr', 'cum_runs_after_pp', 'cum_runs_after_15', 'cum_runs_total', 'wkts_after_pp', 'wkts_after_15', 'wkts_total', 'wkts_rem_after_pp', 'wkts_rem_after_15', 'pp_rpw', 'middle_rpw', 'death_rpw', 'pp_collapse', 'middle_collapse', 'death_collapse', 'accel_pp_to_middle', 'accel_middle_to_death', 'death_share_of_innings', 'pp_aggression', 'middle_aggression', 'death_aggression']

Sample Data (New Features):
   powerplay_rr  middle_rr  death_rr  cum_runs_after_pp  cum_runs_after_15  \
0     10.166667  10.333333      13.6               61.0              154.0   
1      4.333333   6.111111       0.2               26.0               81.0   
2      8.833333  12.000000      15.8               53.0              161.0   
3     10.500000  11.333333       8.4               63.0              165.0   
4      6.666667   6.222222       6.6               40.0               96.0   

   cum_runs_total  wkts_after_pp  wkts_after_15  wkts_total  \

## 7. Venue baselines (training-only)

`v_*_avg` features come from training-era rows only; test-era venues that weren't seen in training fall back to the global training mean. This is the key leakage-prevention step for venue context.


In [9]:
train_only = mdl.loc[train_mask]
v_pp = train_only.groupby("venue")["powerplay_runs"].mean()
v_m  = train_only.groupby("venue")["middle_runs"].mean()
v_d  = train_only.groupby("venue")["death_runs"].mean()
g_pp = train_only["powerplay_runs"].mean()
g_m  = train_only["middle_runs"].mean()
g_d  = train_only["death_runs"].mean()

mdl["v_pp_avg"]     = mdl["venue"].map(v_pp).fillna(g_pp)
mdl["v_middle_avg"] = mdl["venue"].map(v_m).fillna(g_m)
mdl["v_death_avg"]  = mdl["venue"].map(v_d).fillna(g_d)
mdl["v_pp_diff"]     = mdl["powerplay_runs"] - mdl["v_pp_avg"]
mdl["v_middle_diff"] = mdl["middle_runs"]    - mdl["v_middle_avg"]
mdl["v_death_diff"]  = mdl["death_runs"]     - mdl["v_death_avg"]

## 8. Team strength + head-to-head

Rolling team form (career win rate, last-10 form, runs scored / conceded) computed via `groupby(team).shift(1).expanding/rolling` — every stat reflects only matches strictly *before* the current one. Head-to-head uses a canonicalized team-pair key so both perspectives stay consistent.


In [10]:
# Long-form (match × team) frame for rolling stats
ml = pd.concat([
    matches[["match_id", "date", "team1", "winner"]].rename(columns={"team1": "team"}),
    matches[["match_id", "date", "team2", "winner"]].rename(columns={"team2": "team"}),
], ignore_index=True)
ml["won"] = (ml["team"] == ml["winner"]).astype(int)

# Per-(match, team) runs scored / conceded
inn_pair = (df.groupby(["match_id", "batting_team", "bowling_team"])["total_runs"]
              .sum().reset_index())
runs_scored = (inn_pair.groupby(["match_id", "batting_team"])["total_runs"]
                        .sum().reset_index()
                        .rename(columns={"batting_team": "team",
                                         "total_runs":   "runs_scored"}))
runs_conceded = (inn_pair.groupby(["match_id", "bowling_team"])["total_runs"]
                          .sum().reset_index()
                          .rename(columns={"bowling_team": "team",
                                           "total_runs":   "runs_conceded"}))
ml = (ml.merge(runs_scored,   on=["match_id", "team"], how="left")
        .merge(runs_conceded, on=["match_id", "team"], how="left"))
ml = ml.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

# Expanding / rolling team metrics
ml["form_to_date"]         = ml.groupby("team")["won"].transform(lambda s: s.shift(1).expanding().mean())
ml["form_last_10"]         = ml.groupby("team")["won"].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
ml["avg_runs_scored"]      = ml.groupby("team")["runs_scored"].transform(lambda s: s.shift(1).expanding().mean())
ml["avg_runs_conceded"]    = ml.groupby("team")["runs_conceded"].transform(lambda s: s.shift(1).expanding().mean())
ml["recent_runs_scored"]   = ml.groupby("team")["runs_scored"].transform(lambda s: s.shift(1).rolling(5, min_periods=2).mean())
ml["recent_runs_conceded"] = ml.groupby("team")["runs_conceded"].transform(lambda s: s.shift(1).rolling(5, min_periods=2).mean())

# Fill leading NaNs (first appearances) with neutral defaults
for col, val in NEUTRAL_TEAM_STATS.items():
    ml[col] = ml[col].fillna(val)

# Attach as batting-side and bowling-side variants
bat_feats  = ml[["match_id", "team"] + TEAM_COLS].rename(
    columns={"team": "batting_team", **{c: f"bat_{c}"  for c in TEAM_COLS}})
bowl_feats = ml[["match_id", "team"] + TEAM_COLS].rename(
    columns={"team": "bowling_team", **{c: f"bowl_{c}" for c in TEAM_COLS}})
mdl = mdl.merge(bat_feats,  on=["match_id", "batting_team"],  how="left")
mdl = mdl.merge(bowl_feats, on=["match_id", "bowling_team"], how="left")

# Head-to-head — canonicalized pair, expanding win rate
mh = matches.copy().sort_values(["date", "match_id"]).reset_index(drop=True)
mh["pair_a"] = np.where(mh["team1"] < mh["team2"], mh["team1"], mh["team2"])
mh["pair_b"] = np.where(mh["team1"] < mh["team2"], mh["team2"], mh["team1"])
mh["pair_a_won"] = (mh["winner"] == mh["pair_a"]).astype(int)
mh = mh.sort_values(["pair_a", "pair_b", "date", "match_id"]).reset_index(drop=True)
mh["pair_a_h2h_winrate"] = (mh.groupby(["pair_a", "pair_b"])["pair_a_won"]
                              .transform(lambda s: s.shift(1).expanding().mean()))
mh["pair_a_h2h_winrate"] = mh["pair_a_h2h_winrate"].fillna(0.5)
mh["team1_h2h_winrate"] = np.where(mh["pair_a"] == mh["team1"],
                                    mh["pair_a_h2h_winrate"],
                                    1 - mh["pair_a_h2h_winrate"])
mdl = mdl.merge(mh[["match_id", "team1", "team2", "team1_h2h_winrate"]],
                on="match_id", how="left")
mdl["bat_h2h_winrate"] = np.where(mdl["batting_team"] == mdl["team1"],
                                    mdl["team1_h2h_winrate"],
                                    1 - mdl["team1_h2h_winrate"])
mdl = mdl.drop(columns=["team1", "team2", "team1_h2h_winrate"])

In [19]:
# =========================
# ROLLING + H2H FEATURE VALIDATION BLOCK
# =========================

team_feature_cols = (
    [f"bat_{c}" for c in TEAM_COLS] +
    [f"bowl_{c}" for c in TEAM_COLS] +
    ["bat_h2h_winrate"]
)

print("\nCreated Team / H2H Feature Columns:")
print([col for col in team_feature_cols if col in mdl.columns])

print("\nMissing Created Columns:")
print([col for col in team_feature_cols if col not in mdl.columns])

print("\nSample Team Features:")
print(mdl[["match_id", "batting_team", "bowling_team"] + team_feature_cols].head())

print("\nMissing Values in Team / H2H Features:")
print(mdl[team_feature_cols].isnull().sum())

print("\nTeam Feature Summary:")
print(mdl[team_feature_cols].describe())

print("\nHead-to-Head Winrate Check:")
print(mdl["bat_h2h_winrate"].describe())

print("\nBatting Team Rolling Form Sample:")
print(mdl[["batting_team", "bat_form_to_date", "bat_form_last_10"]].head(10))

print("\nBowling Team Rolling Form Sample:")
print(mdl[["bowling_team", "bowl_form_to_date", "bowl_form_last_10"]].head(10))

# =========================
# ASSERTIONS
# =========================

assert all(col in mdl.columns for col in team_feature_cols), "❌ Some team/H2H features were not created"
assert mdl[team_feature_cols].isnull().sum().sum() == 0, "❌ Missing values found in team/H2H features"
assert mdl["bat_h2h_winrate"].between(0, 1).all(), "❌ H2H winrate outside 0-1 range"

for col in team_feature_cols:
    assert np.isfinite(mdl[col]).all(), f"❌ Infinite values found in {col}"

print("\n✅ Rolling team stats + head-to-head checks passed successfully!")


Created Team / H2H Feature Columns:
['bat_form_to_date', 'bat_form_last_10', 'bat_avg_runs_scored', 'bat_avg_runs_conceded', 'bat_recent_runs_scored', 'bat_recent_runs_conceded', 'bowl_form_to_date', 'bowl_form_last_10', 'bowl_avg_runs_scored', 'bowl_avg_runs_conceded', 'bowl_recent_runs_scored', 'bowl_recent_runs_conceded', 'bat_h2h_winrate']

Missing Created Columns:
[]

Sample Team Features:
   match_id                 batting_team                 bowling_team  \
0    335982        Kolkata Knight Riders  Royal Challengers Bangalore   
1    335982  Royal Challengers Bangalore        Kolkata Knight Riders   
2    335983          Chennai Super Kings              Kings XI Punjab   
3    335983              Kings XI Punjab          Chennai Super Kings   
4    335984             Rajasthan Royals             Delhi Daredevils   

   bat_form_to_date  bat_form_last_10  bat_avg_runs_scored  \
0               0.5               0.5                160.0   
1               0.5               0.5 

## 9. Toss + chase pressure

Toss flags, training-only `venue_bat_first_winrate`, and the full chase-pressure stack. Chase features are zero for innings 1 (no target exists yet).

In [11]:
# Toss flags
mdl["batting_won_toss"] = (mdl["batting_team"] == mdl["toss_winner"]).astype(int)
mdl["toss_chose_bat"]   = (mdl["toss_decision"] == "bat").astype(int)

# Venue: P(team batting first wins) — training-era only
first_inn_team = df[df["innings"] == 1].groupby("match_id")["batting_team"].first()
mfm = matches.set_index("match_id")
outcome = pd.DataFrame({
    "venue":          mfm["venue"],
    "first_inn_team": first_inn_team,
    "winner":         mfm["winner"],
}).dropna()
outcome["bat_first_won"] = (outcome["first_inn_team"] == outcome["winner"]).astype(int)
outcome["year"] = mfm["date"].dt.year
train_outcome = outcome[outcome["year"] <= TRAIN_END]
venue_bfw = train_outcome.groupby("venue")["bat_first_won"].mean()
g_bfw     = train_outcome["bat_first_won"].mean()
mdl["venue_bat_first_winrate"] = mdl["venue"].map(venue_bfw).fillna(g_bfw)

# Chase features (innings 2 only — innings 1 stays at neutral zeros)
inn1_totals = mdl[mdl["innings"] == 1].set_index("match_id")["cum_runs_total"]
is_chase    = (mdl["innings"] == 2)
mdl["target_score"]      = mdl["match_id"].map(inn1_totals).where(is_chase, 0).fillna(0)
safe_target              = mdl["target_score"].clip(lower=1)
mdl["chase_progress_pp"] = np.where(is_chase, mdl["cum_runs_after_pp"] / safe_target, 0.0)
mdl["chase_progress_15"] = np.where(is_chase, mdl["cum_runs_after_15"] / safe_target, 0.0)
mdl["req_runs_after_pp"] = np.where(is_chase, mdl["target_score"] - mdl["cum_runs_after_pp"], 0)
mdl["req_runs_after_15"] = np.where(is_chase, mdl["target_score"] - mdl["cum_runs_after_15"], 0)
mdl["req_rr_after_pp"]   = np.where(is_chase, mdl["req_runs_after_pp"] / 14.0, 0)
mdl["req_rr_after_15"]   = np.where(is_chase, mdl["req_runs_after_15"] /  5.0, 0)
mdl["chase_pressure_pp"] = np.where(is_chase, mdl["req_rr_after_pp"]  - mdl["powerplay_rr"], 0)
mdl["chase_pressure_15"] = np.where(is_chase,
                                     mdl["req_rr_after_15"] - (mdl["cum_runs_after_15"] / 15.0),
                                     0)

In [21]:
# =========================
# TOSS + VENUE + CHASE FEATURE VALIDATION BLOCK
# =========================

context_cols = [
    "batting_won_toss",
    "toss_chose_bat",
    "venue_bat_first_winrate",
    "target_score",
    "chase_progress_pp",
    "chase_progress_15",
    "req_runs_after_pp",
    "req_runs_after_15",
    "req_rr_after_pp",
    "req_rr_after_15",
    "chase_pressure_pp",
    "chase_pressure_15"
]

print("\nCreated Context Feature Columns:")
print([col for col in context_cols if col in mdl.columns])

print("\nMissing Created Columns:")
print([col for col in context_cols if col not in mdl.columns])

print("\nSample Context Features:")
print(mdl[["match_id", "innings", "batting_team", "venue"] + context_cols].head(10))

print("\nMissing Values:")
print(mdl[context_cols].isnull().sum())

print("\nFeature Summary:")
print(mdl[context_cols].describe())

print("\nToss Flag Distribution:")
print(mdl[["batting_won_toss", "toss_chose_bat"]].mean())

print("\nVenue Bat First Winrate Summary:")
print(mdl["venue_bat_first_winrate"].describe())

print("\nChase Feature Check by Innings:")
print(mdl.groupby("innings")[[
    "target_score",
    "chase_progress_pp",
    "chase_progress_15",
    "req_rr_after_pp",
    "req_rr_after_15",
    "chase_pressure_pp",
    "chase_pressure_15"
]].describe())

# =========================
# ASSERTIONS
# =========================

assert all(col in mdl.columns for col in context_cols), "Some context features were not created"
assert mdl[context_cols].isnull().sum().sum() == 0, "Missing values found in context features"
assert mdl["venue_bat_first_winrate"].between(0, 1).all(), "Venue winrate outside 0-1 range"

assert mdl.loc[mdl["innings"] == 1, "target_score"].eq(0).all(), "Innings 1 should have target_score = 0"
assert mdl.loc[mdl["innings"] == 1, "chase_progress_pp"].eq(0).all(), "Innings 1 chase_progress_pp should be 0"
assert mdl.loc[mdl["innings"] == 1, "chase_progress_15"].eq(0).all(), "Innings 1 chase_progress_15 should be 0"

assert np.isfinite(mdl[context_cols]).all().all(), "Infinite values found in context features"

print("\n Toss + venue + chase feature checks passed successfully!")


Created Context Feature Columns:
['batting_won_toss', 'toss_chose_bat', 'venue_bat_first_winrate', 'target_score', 'chase_progress_pp', 'chase_progress_15', 'req_runs_after_pp', 'req_runs_after_15', 'req_rr_after_pp', 'req_rr_after_15', 'chase_pressure_pp', 'chase_pressure_15']

Missing Created Columns:
[]

Sample Context Features:
   match_id  innings                 batting_team  \
0    335982        1        Kolkata Knight Riders   
1    335982        2  Royal Challengers Bangalore   
2    335983        1          Chennai Super Kings   
3    335983        2              Kings XI Punjab   
4    335984        1             Rajasthan Royals   
5    335984        2             Delhi Daredevils   
6    335985        1               Mumbai Indians   
7    335985        2  Royal Challengers Bangalore   
8    335986        1              Deccan Chargers   
9    335986        2        Kolkata Knight Riders   

                                        venue  batting_won_toss  \
0             

## 10. Par scores (training-only)

`venue_par_total` and `venue_year_par` use the training-era mean of innings-1 totals at the venue (and venue × year). Score-vs-par deltas reveal *how much* an innings beat or fell short of expectation.

In [12]:
train_inn1 = mdl[(mdl["innings"] == 1) & train_mask]
venue_par_total = train_inn1.groupby("venue")["cum_runs_total"].mean()
venue_year_par  = train_inn1.groupby(["venue", "year"])["cum_runs_total"].mean()
g_par = train_inn1["cum_runs_total"].mean()

mdl["venue_par_total"] = mdl["venue"].map(venue_par_total).fillna(g_par)
vy_idx = pd.MultiIndex.from_arrays([mdl["venue"], mdl["year"]])
mdl["venue_year_par"] = pd.Series(vy_idx.map(venue_year_par).to_numpy(),
                                   index=mdl.index)
mdl["venue_year_par"] = mdl["venue_year_par"].fillna(mdl["venue_par_total"])

venue_par_pp = train_inn1.groupby("venue")["cum_runs_after_pp"].mean()
venue_par_15 = train_inn1.groupby("venue")["cum_runs_after_15"].mean()
mdl["par_after_pp"] = mdl["venue"].map(venue_par_pp).fillna(train_inn1["cum_runs_after_pp"].mean())
mdl["par_after_15"] = mdl["venue"].map(venue_par_15).fillna(train_inn1["cum_runs_after_15"].mean())

mdl["score_vs_venue_par"] = mdl["cum_runs_total"]    - mdl["venue_par_total"]
mdl["score_vs_year_par"]  = mdl["cum_runs_total"]    - mdl["venue_year_par"]
mdl["pp_vs_par"]          = mdl["cum_runs_after_pp"] - mdl["par_after_pp"]
mdl["mid_vs_par"]         = mdl["cum_runs_after_15"] - mdl["par_after_15"]
mdl["overperform_par"]    = (mdl["score_vs_venue_par"] >=  15).astype(int)
mdl["underperform_par"]   = (mdl["score_vs_venue_par"] <= -15).astype(int)

## 11. Momentum + composite indicators

In [13]:
mdl["wicket_intensity"]   = (mdl["wkts_after_pp"]    / 6.0
                              + mdl["middle_wickets"]   / 9.0
                              + mdl["death_wickets"]    / 5.0)
mdl["acceleration_score"] = mdl["accel_pp_to_middle"] + mdl["accel_middle_to_death"]
mdl["momentum_breakdown"] = (mdl["accel_middle_to_death"] < -1.5).astype(int)
mdl["recovery_innings"]   = ((mdl["pp_collapse"] == 1) &
                              (mdl["death_aggression"] > 10)).astype(int)
mdl["stable_innings"]     = ((mdl["wkts_total"] <= 4) &
                              (mdl["cum_runs_total"] >= mdl["venue_par_total"])).astype(int)

In [22]:
# =========================
# STRATEGIC FEATURE VALIDATION BLOCK
# =========================

strategic_cols = [
    "wicket_intensity",
    "acceleration_score",
    "momentum_breakdown",
    "recovery_innings",
    "stable_innings"
]

print("\nCreated Strategic Feature Columns:")
print([col for col in strategic_cols if col in mdl.columns])

print("\nMissing Created Columns:")
print([col for col in strategic_cols if col not in mdl.columns])

print("\nSample Strategic Features:")
print(mdl[[
    "match_id", "innings", "batting_team",
    "wkts_after_pp", "middle_wickets", "death_wickets",
    "accel_pp_to_middle", "accel_middle_to_death",
    "cum_runs_total", "venue_par_total"
] + strategic_cols].head(10))

print("\nMissing Values:")
print(mdl[strategic_cols].isnull().sum())

print("\nFeature Summary:")
print(mdl[strategic_cols].describe())

print("\nFlag Distribution:")
print(mdl[["momentum_breakdown", "recovery_innings", "stable_innings"]].mean())

# =========================
# ASSERTIONS
# =========================

assert all(col in mdl.columns for col in strategic_cols), "Some strategic features were not created"
assert mdl[strategic_cols].isnull().sum().sum() == 0, "Missing values found in strategic features"
assert mdl["wicket_intensity"].min() >= 0, "Negative wicket intensity found"

for col in ["momentum_breakdown", "recovery_innings", "stable_innings"]:
    assert set(mdl[col].unique()).issubset({0, 1}), f"{col} is not binary"

assert np.isfinite(mdl[strategic_cols]).all().all(), "Infinite values found in strategic features"

print("\n Strategic feature checks passed successfully!")


Created Strategic Feature Columns:
['wicket_intensity', 'acceleration_score', 'momentum_breakdown', 'recovery_innings', 'stable_innings']

Missing Created Columns:
[]

Sample Strategic Features:
   match_id  innings                 batting_team  wkts_after_pp  \
0    335982        1        Kolkata Knight Riders              1   
1    335982        2  Royal Challengers Bangalore              4   
2    335983        1          Chennai Super Kings              1   
3    335983        2              Kings XI Punjab              1   
4    335984        1             Rajasthan Royals              2   
5    335984        2             Delhi Daredevils              1   
6    335985        1               Mumbai Indians              3   
7    335985        2  Royal Challengers Bangalore              1   
8    335986        1              Deccan Chargers              2   
9    335986        2        Kolkata Knight Riders              3   

   middle_wickets  death_wickets  accel_pp_to_middle  a

## 12. Final feature list (exact training-time order)

The order below matches the notebook's Phase 3 model exactly:
`PHASE_C_BASE` → `PHASE3_NEW` → `CONTEXT_BASE` → sorted venue dummies.

In [14]:
# Phase block (Section 10 of the analytics notebook)
PHASE_C_BASE = [
    "powerplay_runs", "powerplay_wickets",
    "powerplay_rr", "pp_rpw", "pp_collapse", "pp_aggression",
    "wkts_rem_after_pp", "cum_runs_after_pp", "v_pp_avg", "v_pp_diff",
    "middle_runs", "middle_wickets",
    "middle_rr", "middle_rpw", "middle_collapse", "middle_aggression",
    "wkts_rem_after_15", "cum_runs_after_15", "v_middle_avg", "v_middle_diff",
    "accel_pp_to_middle",
    "death_runs", "death_wickets",
    "death_rr", "death_rpw", "death_collapse", "death_aggression",
    "wkts_total", "cum_runs_total", "death_share_of_innings",
    "v_death_avg", "v_death_diff",
    "accel_middle_to_death",
]

# Strategic block (Section 11 of the analytics notebook)
PHASE3_NEW = [
    # 11.1 team strength
    "bat_form_to_date", "bat_form_last_10",
    "bat_avg_runs_scored", "bat_avg_runs_conceded",
    "bat_recent_runs_scored", "bat_recent_runs_conceded",
    "bowl_form_to_date", "bowl_form_last_10",
    "bowl_avg_runs_scored", "bowl_avg_runs_conceded",
    "bowl_recent_runs_scored", "bowl_recent_runs_conceded",
    "bat_h2h_winrate",
    # 11.2 toss + chase
    "batting_won_toss", "toss_chose_bat", "venue_bat_first_winrate",
    "target_score", "chase_progress_pp", "chase_progress_15",
    "req_runs_after_pp", "req_runs_after_15",
    "req_rr_after_pp", "req_rr_after_15",
    "chase_pressure_pp", "chase_pressure_15",
    # 11.3 par
    "venue_par_total", "venue_year_par",
    "score_vs_venue_par", "score_vs_year_par",
    "overperform_par", "underperform_par",
    "par_after_pp", "par_after_15", "pp_vs_par", "mid_vs_par",
    # 11.4 momentum
    "wicket_intensity", "acceleration_score", "momentum_breakdown",
    "recovery_innings", "stable_innings",
]

CONTEXT_BASE = ["batting_first_or_second", "year", "impact_era_after"]

# Scalar features whose names start with "venue_" but aren't one-hot dummies
VENUE_SCALAR_FEATURES = {"venue_par_total", "venue_year_par",
                          "venue_bat_first_winrate"}

# One-hot encode venue, drop string helpers, lock the feature order
mdl_enc = pd.get_dummies(mdl, columns=["venue"], prefix="venue", drop_first=True)
mdl_enc = mdl_enc.drop(columns=["batting_team", "bowling_team",
                                  "toss_winner", "toss_decision", "date"])

venue_dums = sorted(c for c in mdl_enc.columns
                    if c.startswith("venue_") and c not in VENUE_SCALAR_FEATURES)

feature_columns = PHASE_C_BASE + PHASE3_NEW + CONTEXT_BASE + venue_dums
print(f"Feature count: {len(feature_columns)}")

Feature count: 134


In [23]:
# =========================
# FEATURE COLUMNS + ENCODING VALIDATION BLOCK
# =========================

print("\nFeature count:", len(feature_columns))

print("\nMissing Features:")
missing_features = [c for c in feature_columns if c not in mdl_enc.columns]
print(missing_features)

print("\nDuplicate Features:")
duplicate_features = pd.Series(feature_columns)[pd.Series(feature_columns).duplicated()].tolist()
print(duplicate_features)

print("\nVenue Dummy Count:", len(venue_dums))
print("\nFirst 10 Venue Dummies:")
print(venue_dums[:10])

print("\nFeature Block Counts:")
print("PHASE_C_BASE:", len(PHASE_C_BASE))
print("PHASE3_NEW:", len(PHASE3_NEW))
print("CONTEXT_BASE:", len(CONTEXT_BASE))
print("VENUE_DUMMIES:", len(venue_dums))
print("TOTAL:", len(PHASE_C_BASE) + len(PHASE3_NEW) + len(CONTEXT_BASE) + len(venue_dums))

print("\nEncoded Data Sample:")
print(mdl_enc[feature_columns].head())

print("\nMissing Values in Final Features:")
print(mdl_enc[feature_columns].isnull().sum().sort_values(ascending=False).head(20))

print("\nInfinite Values Check:")
print(np.isinf(mdl_enc[feature_columns].select_dtypes(include=[np.number])).sum().sum())

# =========================
# ASSERTIONS
# =========================

assert len(missing_features) == 0, f"Missing features: {missing_features}"
assert len(duplicate_features) == 0, f"Duplicate features found: {duplicate_features}"
assert mdl_enc[feature_columns].isnull().sum().sum() == 0, "Missing values found in final feature set"
assert np.isfinite(mdl_enc[feature_columns].select_dtypes(include=[np.number])).all().all(), "Infinite values found"
assert "target" not in feature_columns, "Target leakage: target is inside feature_columns"
assert "winner" not in feature_columns, "Target leakage: winner is inside feature_columns"

print("\nAll feature column + encoding checks passed successfully!")


Feature count: 134

Missing Features:
[]

Duplicate Features:
[]

Venue Dummy Count: 58

First 10 Venue Dummies:
['venue_Arun Jaitley Stadium, Delhi', 'venue_Barabati Stadium', 'venue_Barsapara Cricket Stadium, Guwahati', 'venue_Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow', 'venue_Brabourne Stadium', 'venue_Brabourne Stadium, Mumbai', 'venue_Buffalo Park', 'venue_De Beers Diamond Oval', 'venue_Dr DY Patil Sports Academy', 'venue_Dr DY Patil Sports Academy, Mumbai']

Feature Block Counts:
PHASE_C_BASE: 33
PHASE3_NEW: 40
CONTEXT_BASE: 3
VENUE_DUMMIES: 58
TOTAL: 134

Encoded Data Sample:
   powerplay_runs  powerplay_wickets  powerplay_rr  pp_rpw  pp_collapse  \
0            61.0                  1     10.166667    61.0            0   
1            26.0                  4      4.333333     6.5            1   
2            53.0                  1      8.833333    53.0            0   
3            63.0                  1     10.500000    63.0            0   
4     

## 13. Train XGBoost on the chronological training window

In [15]:
X_train = mdl_enc.loc[train_mask, feature_columns]
y_train = mdl_enc.loc[train_mask, "target"]

model = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric="auc",
    tree_method="hist",
)
model.fit(X_train, y_train)
print(f"Trained on {len(X_train):,} innings × {X_train.shape[1]} features")

Trained on 2,010 innings × 134 features


In [24]:
# =========================
# MODEL TRAINING VALIDATION BLOCK
# =========================

print("\nTrain Shape:", X_train.shape)
print("Target Shape:", y_train.shape)

print("\nTarget Distribution:")
print(y_train.value_counts(normalize=True))

print("\nCheck Feature Alignment:")
print("Same index:", X_train.index.equals(y_train.index))

print("\nMissing Values in X_train:")
print(X_train.isnull().sum().sum())

print("\nInfinite Values in X_train:")
print(np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum())

print("\nUnique Classes in Target:")
print(y_train.nunique())

# =========================
# ASSERTIONS
# =========================

assert X_train.shape[0] > 0, "Training data is empty"
assert X_train.shape[1] > 0, "No features found"
assert X_train.index.equals(y_train.index), "Feature and target index mismatch"
assert y_train.nunique() == 2, "Target is not binary"
assert X_train.isnull().sum().sum() == 0, "Missing values in training data"
assert np.isfinite(X_train.select_dtypes(include=[np.number])).all().all(), "Infinite values found in training data"

# =========================
# MODEL FIT CHECK
# =========================

print("\nModel Parameters:")
print(model.get_params())

print("\nFeature Importance (Top 10):")
import pandas as pd
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(feat_imp.head(10))

print("\nAll checks passed. Model trained successfully.")


Train Shape: (2010, 134)
Target Shape: (2010,)

Target Distribution:
target
1    0.5
0    0.5
Name: proportion, dtype: float64

Check Feature Alignment:
Same index: True

Missing Values in X_train:
0

Infinite Values in X_train:
0

Unique Classes in Target:
2

Model Parameters:
{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.85, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'auc', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.05, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 4, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 400, 'n_jobs': -1, 'num_parallel_tree': None, 

## 14. Persist artifacts

In [16]:
with open(MODEL_OUT, "wb") as f:
    pickle.dump(model, f)
with open(FEATURES_OUT, "wb") as f:
    pickle.dump(feature_columns, f)

print(f"Saved {MODEL_OUT} ({os.path.getsize(MODEL_OUT) / 1024:.1f} KB)")
print(f"Saved {FEATURES_OUT} ({len(feature_columns)} features)")

Saved ipl_model.pkl (588.6 KB)
Saved feature_columns.pkl (134 features)
